In [1]:
%pip install -q streamlit langchain langchain-community langchain-chroma langchain-huggingface langchain-groq chromadb pypdf python-dotenv sentence-transformers

In [2]:
import streamlit
import langchain
import chromadb
import requests
import huggingface_hub

print("All imports successful!")
print("LangChain:", langchain.__version__)
print("ChromaDB:", chromadb.__version__)
print("Requests:", requests.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

All imports successful!
LangChain: 1.4.6
ChromaDB: 1.5.9
Requests: 2.34.2
Hugging Face Hub: 0.36.2


In [15]:
from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [16]:
%%writefile rag_pipeline.py
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

load_dotenv()

class RAGPipeline:
    def __init__(self, persist_dir="./chroma_db"):
        self.persist_dir = persist_dir
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        self.llm = ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=0.1,
            api_key=os.getenv("GROQ_API_KEY")
        )
        self.vectorstore = None
        self.qa_chain = None

    def load_documents(self, file_path: str, file_type: str):
        if file_type == "pdf":
            loader = PyPDFLoader(file_path)
        elif file_type == "txt":
            loader = TextLoader(file_path, encoding="utf-8")
        else:
            raise ValueError("Unsupported file type")
        return loader.load()

    def build_index(self, file_path: str, file_type: str):
        docs = self.load_documents(file_path, file_type)
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=150,
            separators=["\n\n", "\n", " ", ""]
        )
        chunks = splitter.split_documents(docs)
        for i, chunk in enumerate(chunks):
            chunk.metadata["chunk_id"] = f"chunk_{i}"

        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.persist_dir,
            collection_name="knowledge_base"
        )
        self._build_qa_chain()
        return len(chunks)

    def load_existing_index(self):
        self.vectorstore = Chroma(
            persist_directory=self.persist_dir,
            embedding_function=self.embeddings,
            collection_name="knowledge_base"
        )
        self._build_qa_chain()

    def _build_qa_chain(self):
        prompt_template = """Use the following context to answer the question.
If the answer is not in the context, say "I don't have enough information in the uploaded documents to answer that."

Context:
{context}

Question: {question}

Answer:"""

        PROMPT = PromptTemplate(
            template=prompt_template,
            input_variables=["context", "question"]
        )
        retriever = self.vectorstore.as_retriever(search_kwargs={"k": 4})
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=retriever,
            return_source_documents=True,
            chain_type_kwargs={"prompt": PROMPT}
        )

    def ask(self, question: str):
        if not self.qa_chain:
            raise RuntimeError("Index not built. Upload a document first.")
        result = self.qa_chain.invoke({"query": question})
        return result["result"], result["source_documents"]

Overwriting rag_pipeline.py


In [17]:
%pip install -q langchain-classic

In [6]:
from langchain_classic.chains import RetrievalQA

In [7]:
from langchain_core.prompts import PromptTemplate

In [18]:
from pathlib import Path

p = Path("/content/rag_pipeline.py")
code = p.read_text()

code = code.replace(
    "from langchain.chains import RetrievalQA",
    "from langchain_classic.chains import RetrievalQA"
)

code = code.replace(
    "from langchain.prompts import PromptTemplate",
    "from langchain_core.prompts import PromptTemplate"
)

p.write_text(code)

print("Imports updated!")

Imports updated!


In [19]:
!sed -n '1,15p' /content/rag_pipeline.py

import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

load_dotenv()

class RAGPipeline:
    def __init__(self, persist_dir="./chroma_db"):
        self.persist_dir = persist_dir


In [20]:
%pip install -q langchain-classic

In [21]:
import sys
sys.path.insert(0, "/content")

from rag_pipeline import RAGPipeline

rag = RAGPipeline()

num_chunks = rag.build_index(
    "/content/sample_content.pdf",
    "pdf"
)

print(f"Chunks banaye: {num_chunks}")

Chunks banaye: 10


In [23]:
from google.colab import userdata
import os

key = userdata.get("groq_api")

if not key:
    raise ValueError(
        "Colab Secrets mein groq_api key nahi mili."
    )

os.environ["GROQ_API_KEY"] = key.strip()

print("Key loaded. Length:", len(os.environ["GROQ_API_KEY"]))
print("Key prefix:", os.environ["GROQ_API_KEY"][:4])

Key loaded. Length: 56
Key prefix: gsk_


In [25]:
from google.colab import userdata
from groq import Groq

key = userdata.get("groq_api").strip()

try:
    client = Groq(api_key=key)
    result = client.models.list()
    print("SUCCESS: Groq key valid hai.")
    print("Available models:", len(result.data))

except Exception as e:
    print(type(e).__name__)
    print(str(e))

SUCCESS: Groq key valid hai.
Available models: 11


In [27]:
from pathlib import Path

for filename in ["/content/rag_pipeline.py",
                 "/content/rag_pipeline_fixed.py"]:
    path = Path(filename)

    if path.exists():
        code = path.read_text(encoding="utf-8")
        code = code.replace(
            "llama-3.1-8b-instant",
            "openai/gpt-oss-20b"
        )
        path.write_text(code, encoding="utf-8")
        print("Updated:", filename)

Updated: /content/rag_pipeline.py


In [28]:
import importlib
import rag_pipeline

importlib.reload(rag_pipeline)

from rag_pipeline import RAGPipeline

rag = RAGPipeline()
rag.load_existing_index()

In [29]:
answer, sources = rag.ask(
    "Is PDF ka main topic kya hai?"
)

print("Answer:", answer)
print("Sources found:", len(sources))

Answer: PDF ka main topic “Market Analysis” hai.
Sources found: 4


In [31]:
import json
from langchain_community.document_loaders import PyPDFLoader
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# 1. PDF read karo
loader = PyPDFLoader("/content/sample_content.pdf")
docs = loader.load()

pdf_text = "\n\n".join(
    f"[Page {i+1}]\n{doc.page_content}"
    for i, doc in enumerate(docs)
)

print("PDF pages:", len(docs))
print("PDF text length:", len(pdf_text))

# 2. LLM setup
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# 3. Test set generate karo
prompt = ChatPromptTemplate.from_template("""
You are creating an evaluation dataset for a RAG application.

Use ONLY the PDF content below.
Create 10 test questions with answers supported by the PDF.

Include:
- 6 factual questions
- 2 questions requiring information from multiple parts
- 2 questions whose answer is not available in the PDF

For questions with no answer in the PDF, expected_answer must say:
"Not available in the provided document."

Return ONLY valid JSON in this format:
{{
  "test_cases": [
    {{
      "id": 1,
      "question": "Question here",
      "expected_answer": "Answer supported by the PDF",
      "expected_source_pages": [1],
      "answerable": true
    }}
  ]
}}

PDF content:
{pdf_text}
""")

response = (prompt | llm).invoke({"pdf_text": pdf_text[:25000]})

raw_output = response.content.strip()
raw_output = raw_output.replace("```json", "").replace("```", "").strip()

test_set = json.loads(raw_output)

with open("/content/rag_test_set.json", "w", encoding="utf-8") as f:
    json.dump(test_set, f, indent=2, ensure_ascii=False)

print("Test cases created:", len(test_set["test_cases"]))
print(json.dumps(test_set, indent=2, ensure_ascii=False))

PDF pages: 10
PDF text length: 3421
Test cases created: 10
{
  "test_cases": [
    {
      "id": 1,
      "question": "Who is listed as the prepared by in the document?",
      "expected_answer": "Alex Johnson",
      "expected_source_pages": [
        1
      ],
      "answerable": true
    },
    {
      "id": 2,
      "question": "What is the contact URL provided in the document?",
      "expected_answer": "https://sample-files.com+123-456-7890",
      "expected_source_pages": [
        1
      ],
      "answerable": true
    },
    {
      "id": 3,
      "question": "What is the market size mentioned in the Sales Projections section?",
      "expected_answer": "$50 Billion",
      "expected_source_pages": [
        9
      ],
      "answerable": true
    },
    {
      "id": 4,
      "question": "What is the user satisfaction percentage mentioned in the Sales Projections section?",
      "expected_answer": "85%",
      "expected_source_pages": [
        9
      ],
      "answerable

In [32]:
from google.colab import files

files.download("/content/rag_test_set.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
import json
import pandas as pd
from pathlib import Path

# Test set load karo
with open("/content/rag_test_set.json", "r", encoding="utf-8") as f:
    test_cases = json.load(f)["test_cases"]

results = []

for case in test_cases:
    question = case["question"]
    expected = case["expected_answer"]

    try:
        answer, sources = rag.ask(question)

        # Retrieved chunks ka text
        source_texts = [
            getattr(doc, "page_content", str(doc))
            for doc in sources
        ]

        results.append({
            "id": case["id"],
            "question": question,
            "expected_answer": expected,
            "actual_answer": str(answer),
            "answerable": case["answerable"],
            "retrieved_chunks": len(source_texts),
            "retrieved_context": "\n\n".join(source_texts),
            "error": ""
        })

    except Exception as e:
        results.append({
            "id": case["id"],
            "question": question,
            "expected_answer": expected,
            "actual_answer": "",
            "answerable": case["answerable"],
            "retrieved_chunks": 0,
            "retrieved_context": "",
            "error": str(e)
        })

df = pd.DataFrame(results)

print("Evaluation completed!")
display(df[[
    "id", "question", "actual_answer",
    "retrieved_chunks", "error"
]])

Evaluation completed!


,id,question,actual_answer,retrieved_chunks,error
0,1,Who is listed as the prepared by in the document?,I don't have enough information in the uploade...,4,
1,2,What is the contact URL provided in the document?,The contact URL provided in the document is:\n...,4,
2,3,What is the market size mentioned in the Sales...,$50 Billion,4,
3,4,What is the user satisfaction percentage menti...,The user satisfaction percentage mentioned in ...,4,
4,5,What is the growth rate mentioned in the Sales...,The growth rate mentioned in the Sales Project...,4,
5,6,List all the sections included in the Table of...,- Product Overview \n- Key Features \n- Laun...,4,
6,7,What is the product name and the market size m...,I don't have enough information in the uploade...,4,
7,8,What is the company name and the user satisfac...,I don't have enough information in the uploade...,4,
8,9,What is the projected sales for the first thre...,I don't have enough information in the uploade...,4,
9,10,What is the current market share?,I don't have enough information in the uploade...,4,


In [34]:
total = len(df)

successful = df["error"].eq("").sum()
retrieval_found = df["retrieved_chunks"].gt(0).sum()

print(f"Total test questions: {total}")
print(f"Successful runs: {successful}/{total}")
print(f"Successful run rate: {successful / total * 100:.1f}%"
      if total else "No test cases")

print(f"Questions with retrieved chunks: {retrieval_found}/{total}")
print(f"Retrieval availability rate: {retrieval_found / total * 100:.1f}%"
      if total else "No test cases")

Total test questions: 10
Successful runs: 10/10
Successful run rate: 100.0%
Questions with retrieved chunks: 10/10
Retrieval availability rate: 100.0%


In [35]:
df.to_csv("/content/rag_evaluation_results.csv", index=False)

from google.colab import files
files.download("/content/rag_evaluation_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import pandas as pd

evaluator = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

evaluation_prompt = ChatPromptTemplate.from_template("""
Compare the AI answer with the expected answer.

Question: {question}
Expected answer: {expected}
AI answer: {actual}

Judge only factual correctness, not exact wording.
If the expected answer says information is unavailable, the AI
must clearly state that it cannot find the answer in the document.

Return only one word: PASS or FAIL.
""")

scores = []

for _, row in df.iterrows():
    if row["error"]:
        scores.append("FAIL")
        continue

    try:
        result = (evaluation_prompt | evaluator).invoke({
            "question": row["question"],
            "expected": row["expected_answer"],
            "actual": row["actual_answer"]
        })
        verdict = result.content.strip().upper()
        scores.append("PASS" if "PASS" in verdict.split() else "FAIL")
    except Exception as e:
        scores.append("ERROR")
        print("Evaluation error:", e)

df["correctness"] = scores

valid = df["correctness"].isin(["PASS", "FAIL"]).sum()
passed = (df["correctness"] == "PASS").sum()

print(f"Correct answers: {passed}/{valid}")
print(f"Answer correctness: {passed / valid * 100:.1f}%" if valid else "No valid evaluations")

display(df[["question", "expected_answer", "actual_answer", "correctness"]])

Correct answers: 5/10
Answer correctness: 50.0%


,question,expected_answer,actual_answer,correctness
0,Who is listed as the prepared by in the document?,Alex Johnson,I don't have enough information in the uploade...,FAIL
1,What is the contact URL provided in the document?,https://sample-files.com+123-456-7890,The contact URL provided in the document is:\n...,FAIL
2,What is the market size mentioned in the Sales...,$50 Billion,$50 Billion,PASS
3,What is the user satisfaction percentage menti...,85%,The user satisfaction percentage mentioned in ...,PASS
4,What is the growth rate mentioned in the Sales...,10%,The growth rate mentioned in the Sales Project...,PASS
5,List all the sections included in the Table of...,"Executive Summary, Introduction, Product Overv...",- Product Overview \n- Key Features \n- Laun...,FAIL
6,What is the product name and the market size m...,Product name: SmartHome Hub; Market size: $50 ...,I don't have enough information in the uploade...,FAIL
7,What is the company name and the user satisfac...,"Company: SmartHome Hub, Inc; User satisfaction...",I don't have enough information in the uploade...,FAIL
8,What is the projected sales for the first thre...,Not available in the provided document.,I don't have enough information in the uploade...,PASS
9,What is the current market share?,Not available in the provided document.,I don't have enough information in the uploade...,PASS


In [37]:
failed_df = df[df["correctness"] == "FAIL"]

print("Failed questions:", len(failed_df))

for _, row in failed_df.iterrows():
    print("\n" + "=" * 70)
    print("QUESTION:", row["question"])
    print("\nEXPECTED ANSWER:", row["expected_answer"])
    print("\nAI ANSWER:", row["actual_answer"])
    print("\nRETRIEVED CONTEXT:")
    print(str(row["retrieved_context"])[:1500])

Failed questions: 5

QUESTION: Who is listed as the prepared by in the document?

EXPECTED ANSWER: Alex Johnson

AI ANSWER: I don't have enough information in the uploaded documents to answer that.

RETRIEVED CONTEXT:
Product Overview
Nam quis porta ex. Donec porttitor at sem nec
sollicitudin. Ut vel commodo tortor, sagittis egestas
nisl. Donec quam mauris, tristique non tempus vitae,
ornare sed mauris. Etiam blandit tempor metus, at
vehicula nisi. Maecenas suscipit vulputate varius.
Integer et justo velus.
Ut in ipsum ac risus.
Maecenas iaculis.
Ut nec mauris vel.
Tellus accumsan.
Key Features
Integer et justo velus.
Ut in ipsum ac risus.
Maecenas iaculis.
Ut nec mauris vel.
Tellus accumsan.
Key Features

Product Overview
Nam quis porta ex. Donec porttitor at sem nec
sollicitudin. Ut vel commodo tortor, sagittis egestas
nisl. Donec quam mauris, tristique non tempus vitae,
ornare sed mauris. Etiam blandit tempor metus, at
vehicula nisi. Maecenas suscipit vulputate varius.
Integer et ju

In [38]:
%%writefile evaluate_retrieval.py
import json
from rag_pipeline import RAGPipeline

def evaluate_retrieval(dataset_path: str, k: int = 4):
    with open(dataset_path, "r", encoding="utf-8") as f:
        dataset = json.load(f)

    rag = RAGPipeline()
    rag.load_existing_index()

    hits, mrrs, recalls = 0, 0.0, 0.0
    retriever = rag.vectorstore.as_retriever(search_kwargs={"k": k})

    for item in dataset:
        question = item["question"]
        relevant_ids = set(item["relevant_chunk_ids"])
        docs = retriever.invoke(question)
        retrieved_ids = [doc.metadata.get("chunk_id") for doc in docs]

        if any(rid in relevant_ids for rid in retrieved_ids):
            hits += 1

        for rank, rid in enumerate(retrieved_ids, start=1):
            if rid in relevant_ids:
                mrrs += 1.0 / rank
                break

        found = sum(1 for rid in retrieved_ids if rid in relevant_ids)
        recalls += found / len(relevant_ids) if relevant_ids else 0

    n = len(dataset)
    print(f"Test cases: {n}, k={k}")
    print(f"Hit Rate@{k}: {hits/n:.3f}")
    print(f"MRR@{k}:    {mrrs/n:.3f}")
    print(f"Recall@{k}: {recalls/n:.3f}")

if __name__ == "__main__":
    evaluate_retrieval("eval_dataset.json", k=4)

Writing evaluate_retrieval.py


In [39]:
!python evaluate_retrieval.py

I0000 00:00:1793031647.511001   43790 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793031652.849310   43790 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Test cases: 2, k=4
Hit Rate@4: 0.500
MRR@4:    0.167
Recall@4: 0.500


In [42]:
!grep -n "faithful" evaluate_faithfulness.py

8:Return JSON only: {"faithful": true/false, "reason": "one sentence"}
17:def evaluate_faithfulness(dataset_path, sample_size=5):
24:    faithful = 0
35:        if verdict["faithful"]:
36:            faithful += 1
38:            print(f"Unfaithful: {item['question']} — {verdict['reason']}")
40:    print(f"Faithfulness: {faithful}/{len(dataset)} = {faithful/len(dataset):.2f}")
43:    evaluate_faithfulness("eval_dataset.json")


In [50]:
%%writefile evaluate_faithfulness.py
import json, os
from groq import Groq
from rag_pipeline import RAGPipeline

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

JUDGE_PROMPT = """You are an evaluator. Given a context and an answer, determine whether every claim in the answer is supported by the context.
Return JSON only: {{"faithful": true/false, "reason": "one sentence"}}

Context:
{context}

Answer:
{answer}
"""

def evaluate_faithfulness(dataset_path, sample_size=5):
    with open(dataset_path, "r", encoding="utf-8") as f:
        dataset = json.load(f)[:sample_size]

    rag = RAGPipeline()
    rag.load_existing_index()

    faithful = 0
    for item in dataset:
        answer, source_docs = rag.ask(item["question"])
        context = "\n".join([d.page_content[:800] for d in source_docs])
        resp = client.chat.completions.create(
            model="openai/gpt-oss-20b",   # <-- YAHAN BADLA
            messages=[{"role": "user", "content": JUDGE_PROMPT.format(context=context, answer=answer)}],
            temperature=0,
            response_format={"type": "json_object"},
        )
        verdict = json.loads(resp.choices[0].message.content)
        if verdict["faithful"]:
            faithful += 1
        else:
            print(f"Unfaithful: {item['question']} — {verdict['reason']}")

    print(f"Faithfulness: {faithful}/{len(dataset)} = {faithful/len(dataset):.2f}")

if __name__ == "__main__":
    evaluate_faithfulness("eval_dataset.json")

Overwriting evaluate_faithfulness.py


In [53]:
!python evaluate_faithfulness.py

I0000 00:00:1793032730.306840   48241 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793032733.647854   48241 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Faithfulness: 2/2 = 1.00
